In [1]:
%%capture

import warnings
warnings.filterwarnings('ignore')

import calitp_data_analysis.magics

In [2]:
import gcsfs as fs
import geopandas as gpd
import numpy as np
import pandas as pd
from calitp_data_analysis import get_fs, utils
from calitp_data_analysis.sql import to_snakecase
from siuba import *

fs = get_fs()

In [3]:
import altair as alt
import folium

In [4]:
import _replica_utils

In [5]:
from IPython.display import HTML

In [6]:
from calitp_data_analysis import calitp_color_palette as cp

In [7]:
pd.set_option("display.max_columns", None)

In [8]:
gcs_path = "gs://calitp-analytics-data/data-analyses/big_data/STM/"


In [9]:
display(HTML("<h1>Origin-Destination Big Data Analysis: SCAG POIs</h1>"))

In [10]:
display(HTML("This analysis uses Replica Data to identify the top trip start points within a Regional Transportation Planning Agency (RTPA) asnd determine what types of trips are occuring when and where."))

In [11]:
shape_data_name = "origins/replica-stm_origin_la-03_06_26-origin_layer.zip"

origins_name = "replica-stm_origin_la-03_06_26-trips_dataset.zip"

In [12]:
origins = _replica_utils.read_in_and_prep_replica_data_w_shp(origins_name, shape_data_name, file_type="")

In [13]:
# len(origins)

In [14]:
display(HTML("<h2><strong>Trip Counts</strong></h2>"))

In [15]:
display(HTML(f"The number of trips Traveling <strong>From</strong> the top 20 locations is <strong>{len(origins)}</strong>"))


In [16]:
assert origins.origin_bgrp_2020.nunique() == 20

In [17]:
origin_tracts_list = origins.origin_bgrp_2020.unique().tolist()

In [18]:
summary = _replica_utils.return_score_summary_single_df(origins, origin_tracts_list, "origin_bgrp_2020")

In [19]:
summary.columns = [col.replace('_', ' ').title() for col in summary.columns]


In [20]:
summary_melt =  pd.melt(
        summary,
        id_vars=["Trip Grouping"],
        value_vars=['Trip Grouping', 'Total Trips',
                    'N Auto Trips', 'Pct Auto Trips',
                    'N Tranist Trips', 'Pct Transit Trips',
                    'N Walking Trips', 'Pct Walking Trips',
                    'Auto Mean Minutes', 'Auto Median Minutes', 'Auto Max Minutes',
                    'Auto Mean Miles','Auto Median Miles', 'Auto Max Miles',
                    'Transit Mean Minutes','Transit Median Minutes', 'Transit Max Minutes',
                    'Transit Mean Miles','Transit Median Miles', 'Transit Max Miles',
                    'Walking Mean Minutes', 'Walking Median Minutes', 'Walking Max Minutes',
                    'Walking Mean Miles','Walking Median Miles', 'Walking Max Miles',
                   ],
        var_name="Metric",  # New column for original measurement names
        # value_name="_Value"
)

In [21]:
display(HTML("<h2><strong>Trip Type Summaries</strong></h2>"))

In [91]:
list_of_dfs = []


for origin in origin_tracts_list:
    origins_subset = origins[origins["origin_bgrp_2020"] == origin]

    modes_breakdown_subset = _replica_utils.get_mode_split(origins_subset, "origin_bgrp_2020")

    list_of_dfs.append(modes_breakdown_subset)

modes_breakdown = pd.concat(list_of_dfs, ignore_index=True)

In [94]:
modes_breakdown.columns = [col.replace('_', ' ').title() for col in modes_breakdown.columns]

In [97]:
modes_breakdown.sample()

,Trip Grouping,Mode,Pct Trips,Total Trips,N Blkgrs Dest,N Blkgrs Origin
134,"1 (Tract 2679.01, Los Angeles, CA)",biking,0.008972,96,3292,1


In [113]:
alt.Chart(modes_breakdown).mark_bar().encode(
    x=alt.X("Trip Grouping:O", title = "Trip Origin"),
    y=alt.Y("Total Trips:Q", title="Total Number of Trips"),
    color=alt.Color("Mode:N", scale=alt.Scale(range=cp.CALITP_CATEGORY_BRIGHT_COLORS)),
    tooltip = ['Total Trips', 'Mode']
    ).properties(
    width=800,  
    height=300,
    title='Modes by Trip Type')

alt.Chart(...)

In [118]:
# alt.Chart(modes_breakdown).mark_bar().encode(
#     x=alt.X('Trip Grouping:O', title ="Trip Origin"),
#     y= alt.Y('Pct Trips:Q', title="Pct of Total Trips"),
#     color=alt.Color('Trip Grouping:N', scale=alt.Scale(range=cp.CALITP_CATEGORY_BRIGHT_COLORS), legend=alt.Legend(title='Trip Grouping')),
#     column= alt.Column('Mode:N', title="Mode"),
#     tooltip=['Pct Trips', 'Mode', 'Trip Grouping']
# ).properties(width = 100, height = 400, title = "Modes by Trip Type")

In [119]:
display(HTML("<h3><strong>Closer Look at Auto, Transit & Walking Trips</strong></h3>"))

In [120]:
display(HTML("<strong>Tip:</strong> Use the dropdown menu to select different metrics to see on the chart."))

In [121]:
metrics_list = summary_melt["Metric"].unique().tolist()

metrics_dropdown = alt.binding_select(
        options=metrics_list,
        name="Metrics: ",
    )
    # Column that controls the bar charts
xcol_param = alt.selection_point(
        fields=["Metric"], value=metrics_list[0], bind=metrics_dropdown
    )

chart = (alt.Chart(summary_melt, title="Metric by Trip Types")
        .mark_bar()
        .encode(
            x=alt.X("value"),
            y=alt.Y("Trip Grouping"),
            color=alt.Color("Trip Grouping",
                                  scale=alt.Scale(
                                      range=cp.CALITP_CATEGORY_BRIGHT_COLORS))
        ).properties(width=400, height=350)
    ).add_params(xcol_param).transform_filter(xcol_param)

display(HTML("""
<style>
form.vega-bindings {
  position: absolute;
  left: 0px;
  top: -0px;
}
</style>
"""))

(chart)

alt.Chart(...)

In [122]:
display(HTML("<br>"
            "<br>"))

In [124]:
summary_long_all_miles = pd.melt(
    summary,
    id_vars=["Trip Grouping"],
    value_vars=[
        'Auto Mean Miles', 'Auto Median Miles', 'Auto Max Miles', 
        'Transit Mean Miles', 'Transit Median Miles', 'Transit Max Miles',
        'Walking Mean Miles', 'Walking Median Miles', 'Walking Max Miles'],
        var_name="Metric",  # New column for original measurement names
        value_name="Miles")

In [125]:
summary_long_all_min = pd.melt(
    summary,
    id_vars=["Trip Grouping"],
    value_vars=[
        'Auto Mean Minutes', 'Auto Median Minutes', 'Auto Max Minutes',
        'Transit Mean Minutes', 'Transit Median Minutes', 'Transit Max Minutes',
        'Walking Mean Minutes', 'Walking Median Minutes', 'Walking Max Minutes'],
        var_name="Metric",  # New column for original measurement names
        value_name="Mintues")

In [126]:
display(HTML("<strong>Trip Length</strong>"))

In [127]:
alt.Chart(summary_long_all_min).mark_bar().encode(
    x="Mintues:Q",
    y="Metric:O",
    color=alt.Color("Metric:N", scale=alt.Scale(range=cp.CALITP_CATEGORY_BRIGHT_COLORS)),
    row="Trip Grouping:O"
).properties(title="Travel Length by Trip Type & Mode")

alt.Chart(...)

In [128]:
alt.Chart(summary_long_all_miles).mark_bar().encode(
    x="Miles:Q",
    y="Metric:O",
    color=alt.Color("Metric:N", scale=alt.Scale(range=cp.CALITP_CATEGORY_BRIGHT_COLORS)),
    row="Trip Grouping:O"
).properties(title="Travel Distance by Trip Type & Mode")

alt.Chart(...)

In [35]:
display(HTML("<h2><strong>Trip Timing</strong></h2>"))

In [129]:
display(HTML("<strong>Tip:</strong> You can zoom into each graph to better see the timing of the trips by mode"))

In [130]:
import datetime

In [132]:
melt_dfs = []
time_duration_dfs =  []

for origin in origin_tracts_list:
    origins_subset = origins[origins["origin_bgrp_2020"] == origin]

    origins_subset_time_melt, origins_subset_time_melt_duration = _replica_utils.return_time_metrics(origins_subset, "trip_start_time", "trip_end_time")
    
    melt_dfs.append(origins_subset_time_melt)
    time_duration_dfs.append(origins_subset_time_melt_duration)



time_melt = pd.concat(melt_dfs, ignore_index=True)
time_duration_melt = pd.concat(time_duration_dfs, ignore_index=True)


In [140]:
alt.Chart(time_melt).mark_circle(size=150).encode(
    x=alt.X('Time:T',axis=alt.Axis(format='%H:%M')),
    y='Metric:O',
    color=alt.Color("Primary Mode"),
    tooltip = ['Primary Mode', 'Metric', 'Time'],
).properties(height=400, width=600, title="Trips Start and End Times").interactive()

alt.Chart(...)

In [41]:
# alt.Chart(time_melt).mark_circle(size=150).encode(
#     x=alt.X('Time:T',axis=alt.Axis(format='%H:%M')),
#     y='Metric:O',
#     color=alt.Color("Primary Mode"),
#     tooltip = ['Primary Mode', 'Metric', 'Time'],
# ).properties(height=400, width=600, title="Trips Start and End Times for Trips To Cal Poly Humboldt").interactive()

In [141]:
alt.Chart(time_duration_melt).mark_circle(size=150).encode(
    x=alt.X('Value:Q', title="Minutes"),
    y='Metric:O',
    color=alt.Color("Primary Mode"),
    tooltip = ['Primary Mode', 'Metric', 'Value']
).properties(height=400, width=600, title="Trip Duration for Trips").interactive()

alt.Chart(...)

In [43]:
# alt.Chart(to_cp_time_melt_duration).mark_circle(size=150).encode(
#     x=alt.X('Value:Q', title="Minutes"),
#     y='Metric:O',
#     color=alt.Color("Primary Mode"),
#     tooltip = ['Primary Mode', 'Metric', 'Value']
# ).properties(height=400, width=600, title="Trip Duration for Trips To Cal Poly Humboldt").interactive()

In [44]:
# n_trips_to_cp = (to_cp>>filter(_.primary_mode != "other_travel_mode")).groupby(["origin_bgrp_2020", "geometry"]).agg(
#         {"activity_id": "nunique"}).reset_index()
# n_trips_to_cp = n_trips_to_cp.set_geometry("geometry")


In [45]:
# n_trips_from_cp = (from_cp>>filter(_.primary_mode != "other_travel_mode")).groupby(["destination_bgrp_2020", "geometry"]).agg(
#         {"activity_id": "nunique"}).reset_index()
# n_trips_from_cp = n_trips_from_cp.set_geometry("geometry")


In [46]:
# n_trips_to_cp['trip_type'] = 'Traveling To Cal Poly'
# n_trips_from_cp['trip_type'] = 'Traveling From Cal Poly'

In [47]:
# ntrips = pd.concat([n_trips_from_cp, n_trips_to_cp])

In [48]:
display(HTML("<h2><strong>Trips by Census Block Groups</strong></h2>"))

In [49]:
# with get_fs().open(f"gs://calitp-analytics-data/data-analyses/traffic_ops/ca_transit_routes.parquet") as f:
#         routes = to_snakecase(gpd.read_parquet(f))

In [50]:
# routes = routes[routes["agency"]=="Humboldt Transit Authority"]

In [51]:
# routes.sample()

In [52]:
# routes.columns = [col.replace('_', ' ').title() for col in routes.columns]


In [53]:
# routes = routes.rename(columns={"Geometry":"geometry", "Shn Route":"SHN Route"})

In [54]:
# minx, miny, maxx, maxy = routes.total_bounds
# sw = [miny, minx]
# ne = [maxy, maxx]

In [55]:
# n_trips_from_cp = n_trips_from_cp.rename(columns={"destination_bgrp_2020":"Census BlockGroup","activity_id":"Number of Trips", "trip_type":"Trip Type"})
# n_trips_to_cp = n_trips_to_cp.rename(columns={"origin_bgrp_2020":"Census BlockGroup", "activity_id":"Number of Trips", "trip_type":"Trip Type"})

In [56]:
display(HTML("<h4>Trips From Cal Poly</h4>"))

In [57]:
# m = n_trips_from_cp.explore(name="Trips from Cal Poly", column="Number of Trips", 
#                 scheme="NaturalBreaks",
#                  k=10,
#                 cmap="YlOrRd")
# m = routes.explore(m=m, column="Route Name", name="HTA Routes", tooltip=["Agency","Route Name", "SHN Route"], cmap="tab20" )
# # this is completely optional
# folium.LayerControl().add_to(m)

# m.fit_bounds([sw, ne])
    
# display(m)

In [58]:
display(HTML("<h3>Trips To Cal Poly</h3>"))

In [59]:
# m = n_trips_to_cp.explore(name="Trips To Cal Poly", column="Number of Trips", 
#                 scheme="NaturalBreaks",
#                  k=10,
#                 cmap="YlOrRd")
# m = routes.explore(m=m, column="Route Name", name="HTA Routes", tooltip=["Agency","Route Name", "SHN Route"])
# # this is completely optional
# folium.LayerControl().add_to(m)

# m

In [60]:
# n_trips_from_cp_mode = (
#     from_cp>>filter(_.primary_mode != "other_travel_mode")).groupby(["destination_bgrp_2020", "geometry", "primary_mode"]).agg(
#     n_trips=("activity_id", "nunique"),
    
#     trip_duration_minutes_median=('trip_duration_minutes', 'median'),
#     trip_duration_minutes_mean=('trip_duration_minutes', 'mean'),
    
#     trip_distance_miles_median=('trip_distance_miles', 'median'),
#     trip_distance_miles_mean=('trip_distance_miles', 'mean'),
    
# #     trip_start_time_mean=('trip_start_time', 'mean'),
# #     trip_start_time_median=('trip_start_time', 'median'),
    
# #     trip_end_time_mean=('trip_end_time', 'mean'),
# #     trip_end_time_median=('trip_end_time', 'median'),

#         ).reset_index()
    
# n_trips_from_cp_mode = n_trips_from_cp_mode.set_geometry("geometry")

In [61]:
# n_trips_to_cp_mode = (
#     to_cp>>filter(_.primary_mode != "other_travel_mode")).groupby(["destination_bgrp_2020", "geometry", "primary_mode"]).agg(
#     n_trips=("activity_id", "nunique"),
    
#     trip_duration_minutes_median=('trip_duration_minutes', 'median'),
#     trip_duration_minutes_mean=('trip_duration_minutes', 'mean'),
    
#     trip_distance_miles_median=('trip_distance_miles', 'median'),
#     trip_distance_miles_mean=('trip_distance_miles', 'mean'),
    
# #     trip_start_time_mean=('trip_start_time', 'mean'),
# #     trip_start_time_median=('trip_start_time', 'median'),
    
# #     trip_end_time_mean=('trip_end_time', 'mean'),
# #     trip_end_time_median=('trip_end_time', 'median'),

#         ).reset_index()
    
# n_trips_to_cp_mode = n_trips_to_cp_mode.set_geometry("geometry")

In [62]:
# n_trips_from_cp_mode.head()

In [63]:
# alt.Chart(n_trips_from_cp_mode).mark_bar().encode(
#     x=alt.X('primary_mode', title ="Trip Mode"),
#     y= alt.Y('n_trips', title="Number of Trips"),
#     # color=alt.Color('destination_bgrp_2020:N', scale=alt.Scale(range=cp.CALITP_CATEGORY_BRIGHT_COLORS), legend=alt.Legend(title='Trip Type')),
# ).properties(width = 500, height = 400, title = "Modes by Trip Type")

In [64]:
# n_trips_from_cp_mode.sample()

In [65]:
# unique_modes = from_cp.primary_mode.unique()

In [66]:
# unique_modes

In [67]:
# unique_modes = ['private_auto', 'public_transit', 'walking']

In [68]:
display(HTML("<h2>Trips Traveling To Cal Poly</h2>"))

In [69]:
# ##hashing out for now for saving
# replica_utils.return_mode_map(n_trips_to_cp_mode, routes, unique_modes, "to")

In [70]:
display(HTML("<h4>Trips Traveling From Cal Poly</h4>"))

In [71]:
###hashing out for now for saving
# replica_utils.return_mode_map(n_trips_from_cp_mode, routes, unique_modes, "from")

In [72]:
display(HTML("<h2>Raw Trip Data Sample</h2>"))

In [73]:
origins.sample(3)

,activity_id,origin_bgrp_2020,origin_trct_2020,origin_cty_2020,origin_st_2020,destination_bgrp_2020,destination_trct_2020,destination_cty_2020,destination_st_2020,primary_mode,trip_purpose,previous_trip_purpose,trip_start_time,trip_end_time,trip_duration_minutes,trip_distance_miles,vehicle_type,vehicle_fuel_type,transit_submode,transit_agency,transit_route,origin_land_use,origin_building_use,destination_land_use,destination_building_use,trip_taker_person_id,trip_taker_household_id,trip_taker_age,trip_taker_sex,trip_taker_race_ethnicity,trip_taker_employment_status,trip_taker_wfh,trip_taker_individual_income,trip_taker_commute_mode,trip_taker_household_size,trip_taker_household_income,trip_taker_available_vehicles,trip_taker_resident_type,trip_taker_industry,trip_taker_building_type,trip_taker_school_grade_attending,trip_taker_education,trip_taker_tenure,trip_taker_language,trip_taker_home_bgrp_2020,trip_taker_home_trct_2020,trip_taker_home_cty_2020,trip_taker_home_st_2020,trip_taker_work_bgrp_2020,trip_taker_work_trct_2020,trip_taker_work_cty_2020,trip_taker_work_st_2020,tour_type,trip_start_local_hour,trip_end_local_hour,origin_zcta_2020,destination_zcta_2020,geometry
155245,15758267171766345490,"1 (Tract 9800, Orange, CA)","9800 (Orange, CA)","Orange County, CA",California,"1 (Tract 91.03, Sacramento, CA)","91.03 (Sacramento, CA)","Sacramento County, CA",California,private_auto,social,recreation,23:02:42,05:33:58,391,409.4,unknown_vehicle_type,other_non_bev,NaN,NaN,NaN,non_retail_attraction,non_retail_attraction,single_family,single_family,8363665793009349914,16095436540102780418,53.0,male,black_not_hispanic_or_latino,employed,remote,30585.0,worked_from_home,4,45878.0,three_plus,core,naics53,single_family,not_attending_school,bachelors_degree,owner,english,"2 (Tract 93.14, Sacramento, CA)","93.14 (Sacramento, CA)","Sacramento County, CA",California,"2 (Tract 93.14, Sacramento, CA)","93.14 (Sacramento, CA)","Sacramento County, CA",California,other_home_based,23,5,92802,95827,"POLYGON ((-117.92842 33.81768, -117.92839 33.8..."
63704,10259659518591588429,"1 (Tract 2077.11, Los Angeles, CA)","2077.11 (Los Angeles, CA)","Los Angeles County, CA",California,"2 (Tract 5734.03, Los Angeles, CA)","5734.03 (Los Angeles, CA)","Los Angeles County, CA",California,private_auto,home,eat,17:37:00,18:28:58,51,26.9,NaN,NaN,NaN,NaN,NaN,retail,retail,single_family,single_family,4163126635493722313,16504134457642015335,62.0,male,hispanic_or_latino_origin,employed,in_person,66420.0,private_auto,2,126112.0,three_plus,core,naics61,single_family,not_attending_school,advanced_degree,owner,english,"2 (Tract 5734.03, Los Angeles, CA)","5734.03 (Los Angeles, CA)","Los Angeles County, CA",California,"1 (Tract 2077.11, Los Angeles, CA)","2077.11 (Los Angeles, CA)","Los Angeles County, CA",California,commute,17,18,90015,90755,"POLYGON ((-118.27266 34.04272, -118.27245 34.0..."
98931,4852892245329896599,"1 (Tract 9800.28, Los Angeles, CA)","9800.28 (Los Angeles, CA)","Los Angeles County, CA",California,"2 (Tract 35.03, Tulare, CA)","35.03 (Tulare, CA)","Tulare County, CA",California,auto_passenger,home,region_arrival,14:03:00,17:23:29,200,163.5,NaN,unknown_fuel_type,NaN,NaN,NaN,transportation_utilities,transportation_utilities,single_family,single_family,3511225928489710355,9337981899910104505,35.0,female,hispanic_or_latino_origin,employed,employed_not_working,12030.0,auto_passenger,10,108782.0,three_plus,core,naics11,single_family,not_attending_school,some_college,owner,spanish,"2 (Tract 35.03, Tulare, CA)","35.03 (Tulare, CA)","Tulare County, CA",California,"1 (Tract 33.01, Tulare, CA)","33.01 (Tulare, CA)","Tulare County, CA",California,undirected,14,17,90045,93257,"POLYGON ((-118.45233 33.94300, -118.43712 33.9..."
